### Per the discussion at the end of the <i><span style="color:red">previous</span></i> <span style="color:blue">Gemini</span> notebook: if you run the two engines against identical prompts, <b><span style="color:orange">ChatGPT</span><span style="color:green"> wins</span></b>... by a <i>longshot</i>.

#### Which makes sense because that prompt was engineered for the <span style="color:orange">chatgpt-4o-mini</span> engine.
> ##### <span style="color:orange">ChatGPT</span> loves incorporating text because it's trained on text. <span style="color:blue">Gemini</span> is a different animal, so after some initial demands (discussed within this notebook), you have to prompt it differently for improved effectiveness.

#### What follows is the notebook to see if <span style="color:blue">Gemini</span> can use all that it learned from  <span style="color:red">YouTube</span> videos and <span style="color:blue">Google Docs</span> to do a better job at resume optimization than <span style="color:orange">OpenAI</span> /  <span style="color:orange">ChatGPT</span>.
***
### Step 1: Imports
> ##### Pretty much the same imports as <span style="color:orange">before</span>, but without '<span style="color:green">WeasyPrint</span>.' To convert Markdown to PDF in this Notebook, the Markdown will be sent to <span style="color:deepskyblue">VS Code</span> for editing, then converted to PDF using the extension '<span style="color:deeppink">Markdown PDF</span>.'
***

In [1]:
# working with our os and .env file
import os
from dotenv import load_dotenv

# since we're using Gemini
import google.generativeai as genai

# make printouts look nicer
from IPython.display import display, Markdown
from markdown import markdown

In [2]:
# load .env stuff - looking for 'True'
load_dotenv()

True

***
### Step 2: Load the Resume
***

In [4]:
# # Sample for REPO:
# resume_file_path = "/Users/nicholasjoseph/Desktop/nj_safe_master_resume.md"

# Sample for PERSONAL USE:
resume_file_path = "/Users/nicholasjoseph/Desktop/nj_master_resume.md"
try:
    with open(resume_file_path, "r", encoding="utf-8") as resume_file:
        resume_string = resume_file.read()
except FileNotFoundError:
    print(f"Error: No file appears to exist at {resume_file_path}")
    exit()

***
### Step 3: Input the Job Description
> ##### Simply copy all the job description details from the website and paste them inside the following box.
***

In [5]:
jd_string = input("Paste the job description here:\n")

Paste the job description here:
 Role: Data Engineer  Location: Remote - US Duration: 06-12 month     Job Description:     Must skills: SQL, DBT, Tableau, and GitHub     Strong proficiency in SQL for data manipulation, querying, and optimization. Hands-on experience with DBT (Data Build Tool) for data transformation and modelling. Experience in building dashboards and data visualizations using Tableau. Proficiency with GitHub for version control and collaborative development. Good understanding of data warehousing concepts and best practices. Strong problem-solving and communication skills. •     Experienced hands-on with Snowflake SQL,DBT, Tableau, GitHub , Oracle HRIS in this order  •     Data engineer with hands-on experience, particularly from a data perspective rather than just system-level knowledge. This is crucial for ensuring they can independently handle tasks and understand the intricacies of the work required.     The essential skills looking for: Snowflake SQL, dbt, and Or

***
### Step 4: Load the API Key
> ##### And insert a reminder to run <span style="color:red">'load_dotenv()'</span> in case there are any problems.
***

In [6]:
gemini_api_key = os.environ.get("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY was not loaded from your environment. Be sure to run the 'load_dotenv()' command.")
genai.configure(api_key=gemini_api_key)

***
### <u>Step 5: Defining the Function Schema</u>
#### <u>First of all, what is a function schema</u>?
##### LLMs lean toward using natural language - free form text. A function schema tells the model (in this case, <span style="color:blue">gemini-2.0-flash</span>) in a JSON-like style the fields you want to make sure you get back and how you want them structured.
##### <b><u>Note</b></u>: you have to do this <i>before</i> you instantiate the model so <span style="color:blue">Gemini</span> knows what format to use and doesn't just fall back on the free form text.
##### The prompt in the original <span style="color:orange">ChatGPT</span> notebook was great for giving information, but it just generated a giant blob of text.
#### <u>Why is a function schema important here</u>?
##### <span style="color:blue">Gemini</span> works a little differently. Instead of just asking for a big block of text, we can send a structured request - we design our response.
#### <u>This is done by</u>:
> ##### a.) Creating a function called 'tailor_resume';
> ##### b.) Telling <span style="color:blue">Gemini</span> what the function does;
> ##### c.) Defining 'parameters' (give us back a JSON object) and 'properties' (fields) like 'tailored_resume' and 'additional_suggestions.'
#### The goal is not only to optimize the content of the resume, but also to get suggestions.
#### Structured requests explicitly state how we want these things back; in the original prompt, we were only <i>hoping</i> to get those things back.
***

In [7]:
function_schema = {
    "name": "tailor_resume",
    "description": "Optimize my resume based on a job description.",

    # 'Here is how we want the info back'
    "parameters": {
        "type": "object",
        "properties": {
            "tailored_resume": {
                "type": "string",
                "description": "The Markdown-formatted resume.",
            },
            "additional_suggestions": {
                "type": "string",
                "description": "Ideas to make your resume stronger for this position.",
            },
        },

        # Make sure we get back SOMETHING
        "required": ["tailored_resume"],
    },
}

***
### Step 6: Create the <span style="color:blue">Gemini</span> Model As An Object (Instantiate It)
> ##### Tell the program to use <span style="color:blue">gemini-2.0-flash</span> and pass the function_schema above
***

In [8]:
model = genai.GenerativeModel(
    model_name = "gemini-2.0-flash",

    # Pass the function_schema for the model to call
    tools = [{"function_declarations" : [function_schema]}]
)

***
### Step 8: Create the Prompt
> ##### Because <span style="color:blue">Gemini</span> is trained differently than <span style="color:orange">ChatGPT</span>, you have to be more explicit with what you want when prompting for <span style="color:blue">Gemini</span>.
***

In [9]:
def prompt_template(resume_text, jd_string):
    return f"""
You are a professional resume optimization expert.
Optimize the resume I have provided you to align with the given job description following the guidelines below:

1. Make the resume one page, relevant, keyword optimized, action-driven.
2. Format it cleanly in **Markdown**.
3. At the end, provide an "**Additional Suggestions**" section with suggestions improvements my resume where gaps exist.

Resume:
{resume_string}

Job Description:
{jd_string}
"""

prompt = prompt_template(resume_string, jd_string)

***
### <u>Step 9: Make the Function Call To Gemini</u>
> ##### Essentially, take the response generated using the roles and tools above and put it into a '<span style="color:red">response</span>' variable.
***

In [10]:
response = model.generate_content(
    contents=[{"role": "user", "parts": [{"text": prompt}]}],
    tools=[{"function_declarations": [function_schema]}],
    generation_config=genai.types.GenerationConfig(temperature=0.7),
)

***
### <u>Step 10: Handle the Response</u>
##### (Quick note about <span style="color:blue">Gemini</span>: Since a single prompt can generate several responses, each one must be assessed for likelihood of being "the best" answer - the one you see. <span style="color:blue">Gemini</span> refers to each of these responses - if there are any responses at all -  as a "<span style="color:blue">candidate</span>" for evaluation. Each <span style="color:blue">candidate</span> is looked at, and the best one is presented to the user.)
#### <u><b><span style="color:purple">Response handling happens in several sub-steps</span></b></u>:
##### 1.) There's a check to see if your prompt actually generates a response;
##### 2.) Each response - each <span style="color:blue">candidate</span> - is looked at and broken down into parts;
##### 3.) Each part ('.parts') holds things like text or function calls; 
##### 4.) Parts are checked for function calls (eg: 'first_part.function_call'); and
##### 5.) If function calls are present, its arguments are extracted (in our case, "tailored_resume" and "additional_suggestions") and assigned their respective variables.
***

In [11]:
# Check to see if candidates were generated 
if response.candidates:
    first_candidate = response.candidates[0]
    
    # Check for content and parts
    if first_candidate.content and first_candidate.content.parts: 
        first_part = first_candidate.content.parts[0]
        
        # Does the response include a function call?
        if first_part.function_call:
            function_call = first_part.function_call
            
            # If so, make sure it's the correct one
            if function_call.name == "tailor_resume":
                
                # Get the arguments from the 'tailor_resume' function call
                arguments = function_call.args
                tailored_resume = arguments.get("tailored_resume")
                additional_suggestions = arguments.get("additional_suggestions")

                # Printouts
                if tailored_resume:
                    print("Tailored Resume:")
                    print(tailored_resume)
                
                # Save resume to this Markdown file for REPO:
                output_file_path = "resumes/nj_repo_gemini_res.md"
                # #Save resume for PERSONAL USE:
                # output_file_path = "/Users/nicholasjoseph/Desktop/nj_gemini_res.md"
                try:
                    with open(saved_markdown_resume, "w", encoding = "utf-8") as output_file:
                        output_file.write(tailored_resume)
                    print(f"Saved Gemini-tailored resume to {output_file_path}")
                except Exception as e:
                    print(f"There's been a problem: we couldn't save the Markdown file: {e}")
                    
                # Some further error-handling
                else:
                    print("Error: tailored_resume not found in function arguments.")
                if additional_suggestions:
                    print("\nAdditional Suggestions:")
                    print(additional_suggestions)
            else:
                print("Function call was made, but not for 'tailor_resume'.")
                print("Function name:", function_call.name)
        elif first_part.text: # Check for text instead
            print("Gemini's direct response:")
            print(first_part.text)
        else:
            print("No function call or text in the response.")
            print(response)
    else:
        print("Response contained no content or parts")
        print(response)
else:
    print("No candidates in response")
    print(response)

Gemini's direct response:
```markdown
## **Nicholas Joseph**
Product Manager | Data Engineer | Tech Enthusiast

Email: [nickpjoseph210@gmail.com](mailto:nickpjoseph210@gmail.com) | Phone: (210) 771-9853 | LinkedIn: [nickjoseph](https://www.linkedin.com/in/nickjoseph) | GitHub: [github.com/npj210mlk](https://github.com/npj210mlk)

### Summary

Data Engineer with 3+ years of experience in data pipeline development and a proven track record of delivering data-driven solutions. Expertise in SQL, dbt, Tableau, and GitHub. Adept at data transformation, modeling, and building insightful dashboards. Strong understanding of data warehousing concepts and Oracle HRIS data structures. Passionate about leveraging data to drive business decisions and improve overall efficiency.

### Skills

**Technical:** SQL, dbt, Tableau, GitHub, Snowflake, Oracle HRIS, Data Warehousing, Data Modeling, Data Transformation, Data Analysis & Visualization, Python, Cloud Computing (Azure, GCP), Google BigQuery, Databr